# Task 8 – Collaborative Filtering Recommendation Model

**Objective:** Recommend TED Talks to a user based on what *similar users* liked (User-Based Collaborative Filtering).

> **Note:** The user interaction data used here is **simulated** (created in Task 7). It is for learning purposes only and does not come from real TED viewers.

In [1]:
import pandas as pd                                    # tables and data handling
import numpy as np                                     # numbers and maths helpers
from sklearn.metrics.pairwise import cosine_similarity # measures how alike two users are

In [2]:
# Step 1: load the Task 7 interaction data and the original TED talks (only the 2 columns we need)
interactions = pd.read_csv("../data/interaction/user_interactions.csv")
talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv", usecols=["talk_id", "title"])

print("Interaction data shape:", interactions.shape)
print("Columns:", list(interactions.columns))
print("\nFirst 5 rows:")
display(interactions.head())
print("\nTalks data shape:", talks.shape)
display(talks.head(3))
print("\nMissing values in interactions:")
print(interactions.isnull().sum())

Interaction data shape: (181, 7)
Columns: ['user_id', 'talk_id', 'watched', 'rating', 'reaction', 'saved', 'data_source']

First 5 rows:


,user_id,talk_id,watched,rating,reaction,saved,data_source
0,1,2405,1,5.0,like,0,simulated
1,1,2577,1,5.0,like,0,simulated
2,1,2217,1,4.0,like,0,simulated
3,1,2774,1,5.0,like,0,simulated
4,1,206,1,5.0,like,1,simulated



Talks data shape: (4005, 2)


,talk_id,title
0,1,Averting the climate crisis
1,92,The best stats you've ever seen
2,7,Simplicity sells



Missing values in interactions:
user_id         0
talk_id         0
watched         0
rating         17
reaction        0
saved           0
data_source     0
dtype: int64


In [3]:
# Your Task 7 file might call the columns slightly different names, so this finds them automatically
def find_col(df, options):
    for c in df.columns:
        if c.lower().strip().replace(" ", "_") in options:
            return c
    raise KeyError(f"None of {options} found. Columns are: {list(df.columns)}")

user_col   = find_col(interactions, ["user_id", "userid", "user"])
talk_col   = find_col(interactions, ["talk_id", "ted_talk_id", "talkid", "talk"])
rating_col = find_col(interactions, ["rating", "ratings", "score"])

# rename to simple standard names used in the rest of the notebook
interactions = interactions.rename(columns={user_col: "user_id", talk_col: "talk_id", rating_col: "rating"})

# make sure IDs are the same type in both tables so they match when we merge
interactions["talk_id"] = pd.to_numeric(interactions["talk_id"])
talks["talk_id"] = pd.to_numeric(talks["talk_id"])

# check that every talk ID in the interactions exists in the TED dataset
missing_ids = set(interactions["talk_id"]) - set(talks["talk_id"])
print("Talk IDs in interactions but NOT in TED dataset:", len(missing_ids))
print("Users:", interactions["user_id"].nunique(), "| Talks rated:", interactions["talk_id"].nunique())

Talk IDs in interactions but NOT in TED dataset: 0
Users: 24 | Talks rated: 24


## 1. User-Item Interaction Matrix
Rows = users, columns = TED Talks, values = ratings. An empty cell (NaN) means the user has **not rated** that talk.

In [4]:
# pivot: one row per user, one column per talk, cell = rating (average if a user rated the same talk twice)
user_item_matrix = interactions.pivot_table(index="user_id", columns="talk_id", values="rating", aggfunc="mean")

print("Matrix shape (users x talks):", user_item_matrix.shape)
total_cells = user_item_matrix.shape[0] * user_item_matrix.shape[1]
print("Percentage of empty cells (sparsity):", round(user_item_matrix.isnull().sum().sum() / total_cells * 100, 2), "%")

# show only the 10 most-rated talks so the screenshot is not full of empty cells
top_talks = user_item_matrix.notna().sum().sort_values(ascending=False).head(10).index
display(user_item_matrix[top_talks].head(10))

Matrix shape (users x talks): (24, 24)
Percentage of empty cells (sparsity): 71.53 %


talk_id,263,1272,24138,1344,2399,2774,1733,2577,3136,32560
user_id,,,,,,,,,,
1,NaN,3.0,NaN,NaN,NaN,5.0,NaN,5.0,1.0,NaN
2,NaN,NaN,NaN,NaN,NaN,4.0,NaN,5.0,4.0,NaN
3,2.0,NaN,NaN,2.0,NaN,5.0,4.0,5.0,NaN,NaN
4,NaN,NaN,3.0,NaN,NaN,NaN,NaN,4.0,4.0,NaN
5,1.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,2.0,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN,NaN
7,NaN,5.0,4.0,NaN,NaN,NaN,5.0,NaN,NaN,NaN
8,NaN,4.0,3.0,NaN,1.0,NaN,5.0,2.0,NaN,1.0
9,NaN,4.0,4.0,2.0,NaN,NaN,5.0,NaN,NaN,NaN


## 2. User Similarity Matrix (Cosine Similarity)
For this first version, missing ratings are filled with **0**. Zero here only means *"no interaction data"*, **not** an actual rating of zero.

In [5]:
# fill empty cells with 0 (simplified approach, as the task says)
user_item_filled = user_item_matrix.fillna(0)

# cosine similarity between every pair of users
similarity_array = cosine_similarity(user_item_filled)

# put it in a table with user IDs as row and column names
user_similarity_df = pd.DataFrame(similarity_array,
                                  index=user_item_matrix.index,
                                  columns=user_item_matrix.index)

print("Similarity matrix shape:", user_similarity_df.shape)
display(user_similarity_df.iloc[:10, :10].round(2))

Similarity matrix shape: (24, 24)


user_id,1,2,3,4,5,6,7,8,9,10
user_id,,,,,,,,,,
1,1.00,0.78,0.71,0.33,0.31,0.64,0.15,0.31,0.13,0.20
2,0.78,1.00,0.78,0.52,0.52,0.77,0.00,0.32,0.07,0.16
3,0.71,0.78,1.00,0.27,0.25,0.78,0.19,0.44,0.22,0.26
4,0.33,0.52,0.27,1.00,0.16,0.34,0.20,0.25,0.17,0.00
5,0.31,0.52,0.25,0.16,1.00,0.23,0.00,0.29,0.19,0.06
6,0.64,0.77,0.78,0.34,0.23,1.00,0.00,0.32,0.05,0.15
7,0.15,0.00,0.19,0.20,0.00,0.00,1.00,0.59,0.80,0.66
8,0.31,0.32,0.44,0.25,0.29,0.32,0.59,1.00,0.77,0.59
9,0.13,0.07,0.22,0.17,0.19,0.05,0.80,0.77,1.00,0.77


## 3. Identify Similar Users

In [6]:
def get_similar_users(user_id, n=5):
    # take this user's similarity scores, drop the user themselves, sort highest first
    scores = user_similarity_df[user_id].drop(user_id)
    return scores.sort_values(ascending=False).head(n)

selected_user = user_item_matrix.index[0]   # first user in the data
print("Selected user:", selected_user)
print("\nMost similar users:")
print(get_similar_users(selected_user, n=5).round(3))

Selected user: 1

Most similar users:
user_id
2     0.784
3     0.706
6     0.637
17    0.370
4     0.333
Name: 1, dtype: float64


## 4. Recommendation Function
**Predicted rating** = a weighted average of the ratings given by similar users. Users with higher similarity count more.

predicted score = sum(similarity × rating) / sum(similarity), using only similar users who actually rated that talk.

In [7]:
def recommend_for_user(user_id, top_n=5, n_neighbors=10):
    # 1. handle an invalid user ID
    if user_id not in user_item_matrix.index:
        print(f"User ID '{user_id}' not found. Please enter a valid user ID.")
        return pd.DataFrame(columns=["talk_id", "title", "predicted_rating"])

    # 2. find the most similar users (the user themselves is excluded inside get_similar_users)
    similar_users = get_similar_users(user_id, n_neighbors)
    similar_users = similar_users[similar_users > 0]   # keep only users with some similarity
    if similar_users.empty:
        print(f"No similar users found for '{user_id}'.")
        return pd.DataFrame(columns=["talk_id", "title", "predicted_rating"])

    # 3. talks this user has already seen = rated OR watched (so we never recommend them again)
    watched_talks = interactions[(interactions["user_id"] == user_id) & (interactions["watched"] == 1)]["talk_id"]
    already_seen = set(user_item_matrix.loc[user_id].dropna().index) | set(watched_talks)
    candidate_talks = [t for t in user_item_matrix.columns if t not in already_seen]

    # 4. ratings that similar users gave to those candidate talks
    neighbor_ratings = user_item_matrix.loc[similar_users.index, candidate_talks]

    # 5. weighted average: (similarity x rating) added up, divided by the similarities of users who rated
    weighted_sum = neighbor_ratings.fillna(0).mul(similar_users, axis=0).sum()
    similarity_sum = neighbor_ratings.notna().mul(similar_users, axis=0).sum()
    predicted = (weighted_sum / similarity_sum).dropna()   # talks nobody similar rated are dropped

    # 6. sort highest first and keep the top N
    top = predicted.sort_values(ascending=False).head(top_n)

    # 7. attach the TED talk titles
    result = top.reset_index()
    result.columns = ["talk_id", "predicted_rating"]
    result = result.merge(talks, on="talk_id", how="left")
    result["predicted_rating"] = result["predicted_rating"].round(2)
    return result[["talk_id", "title", "predicted_rating"]]

## 5. Testing the Model

In [8]:
test_users = user_item_matrix.index[:3].tolist()   # three different users
results = {}

for u in test_users:
    print(f"\n=== Recommendations for user: {u} ===")
    results[u] = recommend_for_user(u, top_n=5)
    display(results[u])

print("\n----- AUTOMATIC CHECKS -----")
for u, rec in results.items():
    watched_u = interactions[(interactions["user_id"] == u) & (interactions["watched"] == 1)]["talk_id"]
    seen = set(user_item_matrix.loc[u].dropna().index) | set(watched_u)
    no_seen = len(set(rec["talk_id"]) & seen) == 0
    is_sorted = rec["predicted_rating"].is_monotonic_decreasing
    print(f"User {u}: got {len(rec)} recs | no already-seen talks: {no_seen} | sorted high to low: {is_sorted}")

unique_lists = {tuple(r["talk_id"]) for r in results.values()}
print("Different users got different recommendations:", len(unique_lists) > 1)

print("\n--- Invalid user test ---")
_ = recommend_for_user("NOT_A_USER")
_ = recommend_for_user(99999)


=== Recommendations for user: 1 ===


,talk_id,title,predicted_rating
0,2476,Your kids might live on Mars. Here's how they'...,5.00
1,685,The thrilling potential of SixthSense technology,4.06
2,66,Do schools kill creativity?,4.00
3,1042,The power of vulnerability,3.55
4,2452,This country isn't just carbon neutral -- it's...,3.45



=== Recommendations for user: 2 ===


,talk_id,title,predicted_rating
0,2361,How to stay calm when you know you'll be stressed,4.00
1,66,Do schools kill creativity?,3.80
2,1733,Grit: The power of passion and perseverance,3.56
3,1042,The power of vulnerability,3.51
4,1272,A 12-year-old app developer,3.04



=== Recommendations for user: 3 ===


,talk_id,title,predicted_rating
0,66,Do schools kill creativity?,4.18
1,1386,Questions no one knows the answers to,4.00
2,206,Underwater astonishments,3.83
3,1042,The power of vulnerability,3.52
4,2361,How to stay calm when you know you'll be stressed,3.50



----- AUTOMATIC CHECKS -----
User 1: got 5 recs | no already-seen talks: True | sorted high to low: True
User 2: got 5 recs | no already-seen talks: True | sorted high to low: True
User 3: got 5 recs | no already-seen talks: True | sorted high to low: True
Different users got different recommendations: True

--- Invalid user test ---
User ID 'NOT_A_USER' not found. Please enter a valid user ID.
User ID '99999' not found. Please enter a valid user ID.


In [9]:
# A brand new user has no history, so similarity is impossible. A simple fallback: most popular talks.
def popular_talks(top_n=5, min_ratings=3):
    stats = interactions.groupby("talk_id")["rating"].agg(["mean", "count"])
    stats = stats[stats["count"] >= min_ratings].sort_values("mean", ascending=False).head(top_n)
    stats = stats.reset_index().merge(talks, on="talk_id", how="left")
    stats["mean"] = stats["mean"].round(2)
    return stats.rename(columns={"mean": "average_rating", "count": "num_ratings"})[["talk_id", "title", "average_rating", "num_ratings"]]

print("New user (no history) fallback recommendations:")
display(popular_talks())

New user (no history) fallback recommendations:


,talk_id,title,average_rating,num_ratings
0,1386,Questions no one knows the answers to,4.5,4
1,2452,This country isn't just carbon neutral -- it's...,4.2,5
2,2405,This is what happens when you reply to spam email,4.0,6
3,2577,The agony of trying to unsubscribe,4.0,8
4,2558,How trees talk to each other,4.0,5


## Questions and Answers

**1. What is Collaborative Filtering?**
A way of recommending items by looking at what *other people with similar taste* liked, instead of looking at the item's content.

**2. How is it different from Content-Based Filtering?**
Content-based asks "which talks have similar text to this talk?" Collaborative asks "which users behave like this user, and what did they enjoy?" The first uses talk content, the second uses user behaviour.

**3. What is a User-Item Interaction Matrix?**
A table where each row is a user, each column is a TED Talk, and each cell is the rating that user gave. Empty cells mean not rated.

**4. Why do we calculate similarity between users?**
To find "taste twins". Their ratings on talks I haven't seen are the best clue about what I may enjoy.

**5. What does a cosine similarity score represent?**
How closely two users' rating patterns point in the same direction. 1 means very similar, 0 means nothing in common. Since ratings are positive, scores here fall between 0 and 1.

**6. Why exclude TED Talks already rated by a user?**
Recommending something they have already seen is useless. We want new discoveries. In my code I exclude talks that were rated or watched.

**7. What is a predicted rating?**
The rating we estimate the user would give a talk they haven't seen, calculated as the similarity-weighted average of similar users' ratings.

**8. What happens when a new user has no interaction history?**
They have no row of ratings, so we cannot find similar users and cannot recommend. We need a fallback such as popular talks or content-based recommendations.

**9. What is the Cold-Start Problem?**
The problem of recommending to a new user, or recommending a new talk, when there is no interaction data yet.

**10. What are the limitations of User-Based Collaborative Filtering?**
- Cold-start for new users and new talks
- Sparse data: most cells in the matrix are empty (71.53% in my data)
- Filling missing values with 0 can look like a "dislike"
- Gets slower as users grow, because every user is compared with every other user
- Cannot recommend talks that nobody has rated
- Simulated data does not reflect real human behaviour

## Observations

*(All interaction data is simulated, as marked by the `data_source` column.)*

- **How similar users are identified:** Each user is a row of ratings in the user-item matrix. Cosine similarity compares two rows, and the highest scores (excluding the user themselves) show similar users. For user 1, the most similar users were 2 (0.78), 3 (0.71) and 6 (0.64). The matrix also showed two clear groups: users 1, 2, 3, 6 and users 7, 8, 9, 10.
- **How recommendations are generated:** For talks the user has not seen, I took the ratings of similar users and calculated a similarity-weighted average. The five highest predicted ratings became the recommendations.
- **Different users, different results:** The checks confirmed that users 1, 2 and 3 received different lists. Some popular talks, such as "Do schools kill creativity?" and "The power of vulnerability", appeared for more than one user because many of their neighbours rated them highly.
- **How preferences influence results:** A user's ratings decide who their neighbours are, and the neighbours' ratings decide the recommendations. Changing a user's ratings would change their neighbours and their list.
- **Limitations observed:**
  - The matrix is 71.53% empty (sparse).
  - Filling missing values with 0 makes "not rated" look like a rating.
  - Some predicted scores may depend on only one or two neighbours, so they can be unreliable. For example, a score of exactly 5.00 may come from a single neighbour's rating.
  - Users with no history (cold start) get nothing from this model.
  - The data is small and simulated, so real-world behaviour may differ.
- **Next step:** Combine this with the content-based model into a Hybrid model.